In [ ]:
#r "..\src\functions\bin\Debug\net9.0\functions.dll"
#r "nuget: Microsoft.Extensions.Logging, 9.0.0"
//#!import "../src/functions/Azure/ContentUnderstanding/AzureContentUnderstandingClient.cs"

using Microsoft.Extensions.Logging;
using Billy.Function.AzureContentUnderstanding;
using Billy.Function.Models;
using Billy.Function.Models.ACM;
using Billy.Function.Parsing;
using System.Text.Json;

var endpoint = "https://billyaiservices783743165220.cognitiveservices.azure.com";
var apiVersion = "2024-12-01-preview";
var subscriptionKey = Environment.GetEnvironmentVariable("ACU_SUBSCRIPTION_KEY");
var apiToken = Environment.GetEnvironmentVariable("ACU_API_TOKEN") ?? "";


var loggerFactory = LoggerFactory.Create(builder =>
{
    builder.AddConsole(options =>
    {
        options.FormatterName = "Simple";
    }).SetMinimumLevel(LogLevel.Information);
    builder.AddSimpleConsole(options =>
    {
        options.SingleLine = true;
        options.IncludeScopes = false;
        options.UseUtcTimestamp = false;
        options.TimestampFormat = "yyyy-MM-dd HH:mm:ss ";
    });
    // builder.Configure(options =>
    // {
    //     options.ActivityTrackingOptions = ActivityTrackingOptions.None;
    // });
});
var _logger = loggerFactory.CreateLogger<AzureContentUnderstandingClient>();

var client = new AzureContentUnderstandingClient(_logger, endpoint, apiVersion, subscriptionKey, apiToken);



Installed Packages Microsoft.Extensions.Logging, 9.0.0

### Test Invoices

In [3]:
var responseMessage = await client.BeginAnalyzeAsync("BillAnalyzer", "../res/invoices/DELAPAZ.jpg");
// _logger.LogInformation("Response message: {responseMessage}", responseMessage.ToString());
var result = await client.PollResultAsync(responseMessage);
// JsonSerializer.Serialize(result, new JsonSerializerOptions { WriteIndented = true })
var json = client.GetJsonFields(result);
Invoice invoice  = GenericParser.ParseJson<Invoice>(json);
_logger.LogInformation($"Invoice Details:");
_logger.LogInformation($"Customer: {invoice.CustomerName}");
_logger.LogInformation($"Amount Due: {invoice.AmountDue}");
_logger.LogInformation($"Invoice Date: {invoice.InvoiceDate:yyyy-MM-dd}");
_logger.LogInformation($"Due Date: {invoice.DueDate:yyyy-MM-dd}");
_logger.LogInformation($"Total Items: {invoice.Items?.Count ?? 0}");
Console.WriteLine($"TOTAL: {invoice.InvoiceTotal}");

// Display item details if available
if (invoice.Items != null && invoice.Items.Count > 0)
{
    _logger.LogInformation("\nItem Details:");
    foreach (var item in invoice.Items)
    {
        _logger.LogInformation($"- {item.Description}: {item.TotalPrice}");
    }
}

"FINISHED"


TOTAL: 77.7


FINISHED

### Test Front Debit/Credit Card

In [3]:
var responseMessage = await client.BeginAnalyzeAsync("FrontDebitCardAnalyzer", "../res/cards/chase_front.jpg");
// _logger.LogInformation("Response message: {responseMessage}", responseMessage.ToString());
var result = await client.PollResultAsync(responseMessage);
// JsonSerializer.Serialize(result, new JsonSerializerOptions { WriteIndented = true });
var json = client.GetJsonFields(result);

Card card = GenericParser.ParseJson<Card>(json);
Console.WriteLine($"Name: {card.Name}");
Console.WriteLine($"Card Number: {card.CardNumber}");
Console.WriteLine($"Card Issuer: {card.CardIssuer}");
Console.WriteLine($"Expiration: {card.GoodThru}");

Name: MARCO MEDRANO
Card Number: 4347697015322211
Card Issuer: VISA
Expiration: 10/24


### Test Back Debit/Credit Card

In [2]:
var responseMessage = await client.BeginAnalyzeAsync("BackDebitCardAnalyzer", "../res/cards/chase_back.jpg");
// _logger.LogInformation("Response message: {responseMessage}", responseMessage.ToString());
var result = await client.PollResultAsync(responseMessage);
// JsonSerializer.Serialize(result, new JsonSerializerOptions { WriteIndented = true })
var json = client.GetJsonFields(result);

Card card = GenericParser.ParseJson<Card>(json);
Console.WriteLine($"SecurityCode: {card.SecurityCode}");

SecurityCode: 249
